# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² clinical dataset using the `mlcroissant` library.

### Dataset Source
Structured Croissant schema at [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json).

In [ ]:
# Ensure `mlcroissant` and plot libraries are installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset summary info
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)

## 2. Data Overview
Review available record sets, their IDs, fields, and columns defined in the Croissant schema.

In [ ]:
# List all record sets and their @ids
# Fetch the Croissant parsed recordSet objects
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for f in fields:
                print(f"  Field @id: {f['@id']} (label: {f.get('label', f['@id'])})")
        if 'column' in rs:
            columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
            for c in columns:
                print(f"  Column @id: {c['@id']} (label: {c.get('label', c['@id'])})")

## 3. Data Extraction
Load data from each record set into dataframes for analysis. All entities are referenced by their `@id` as per Croissant schema.

In [ ]:
# Prepare extraction of all available record sets by @id
record_set_ids = []
for rs in (dataset.metadata.recordSet or []):
    record_set_ids.append(rs['@id'])
print(f"All Record Set @ids: {record_set_ids}")

# Extract each record set to a dataframe
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded dataframe for {rs_id} with columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head())
    else:
        print(f"No records found for {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group numeric columns using the `@id` of the fields.

**Note:** Replace the placeholders below with actual field/column `@id`s from Section 2, depending on the record set and columns available.

In [ ]:
# Example: Perform EDA on the first available record set
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes.get(rs_id)
    if df is not None:
        # Find candidate numeric fields using Croissant metadata
        fields = next((rs['field'] for rs in dataset.metadata.recordSet if rs['@id'] == rs_id), None)
        fields = fields if isinstance(fields, list) else [fields] if fields else []

        numeric_field_ids = []
        for f in fields:
            dt = f.get('dataType', None)
            if dt in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_field_ids.append(f['@id'])

        if numeric_field_ids:
            numeric_field = numeric_field_ids[0]  # Select first numeric field
            if numeric_field in df.columns:
                threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
                filtered_df = df[df[numeric_field] > threshold]
                print(f"Filtered records with {numeric_field} > {threshold}:")
                print(filtered_df.head())

                # Normalize
                filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
                print(f"Normalized {numeric_field} for filtered records:")
                print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

                # Group by categorical field if any
                group_field = None
                for f in fields:
                    dt = f.get('dataType', '')
                    if dt not in ['schema:Integer', 'schema:Float', 'schema:Number']:
                        if f['@id'] in df.columns:
                            group_field = f['@id']
                            break
                if group_field:
                    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                    print(f"Grouped data by {group_field}:")
                    print(grouped_df.head())
                else:
                    print("No categorical group field found.")
            else:
                print(f"Numeric field {numeric_field} not found in columns.")
        else:
            print("No numeric fields available in the first record set.")
    else:
        print(f"No dataframe found for record set {rs_id}.")

## 5. Visualization
Visualize distributions or relationships using matplotlib/seaborn. This assumes there are numeric fields in the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric fields for first record set if present
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes.get(rs_id)
    if df is not None and numeric_field_ids:
        numeric_field = numeric_field_ids[0]
        if numeric_field in df.columns:
            plt.figure(figsize=(6, 4))
            sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
            plt.title(f"Distribution of {numeric_field}")
            plt.xlabel(numeric_field)
            plt.ylabel('Frequency')
            plt.show()

            # If normalized field exists
            norm_field = f"{numeric_field}_normalized"
            if norm_field in df.columns:
                plt.figure(figsize=(6, 4))
                sns.histplot(df[norm_field].dropna(), bins=10, kde=True)
                plt.title(f"Normalized Distribution of {numeric_field}")
                plt.xlabel(norm_field)
                plt.ylabel('Frequency')
                plt.show()
        else:
            print(f"Numeric field {numeric_field} not present in dataframe.")
    else:
        print("Unable to visualize: DataFrame or numeric fields missing.")

## 6. Conclusion
This notebook provided an overview and extraction process for the FAIR² clinical dataset using the `mlcroissant` library.

- Used Croissant schema referencing entities by their `@id` fields for full traceability and reproducibility.
- Record sets, fields, and columns were explored and loaded dynamically.
- Demonstrated filtering, normalization, grouping, and visualization using the actual Croissant structure.
- Remember, this dataset is for academic and clinical illustration only, due to its small sample size and specific cohort.

For more advanced analysis, explore record set relationships, field attributes, and Croissant metadata using their `@id` references, and document all steps for optimal reproducibility.